# European Equity Option Pricing Demo

This notebook demonstrates how to price a European Equity Option using the `atlas` library.

We show two approaches:
1. **Pricing Dispatcher**: Constructing a `MarketDataSnapshot` and a `EuropeanEquityOption` instrument and using the pricing dispatcher (`calculate_price`).
2. **Direct BSM Pricer**: Directly using the Black-Scholes-Merton option pricing function (`black_scholes_merton_pricer`).

## Method 1: Using the Pricing Dispatcher

In this method, we construct standard domain objects: `EuropeanEquityOption` and `MarketDataSnapshot`, and call the central dispatcher function `calculate_price`.

In [ ]:
import QuantLib as ql
from atlas.domain.instruments.equities.options import EuropeanEquityOption
from atlas.domain.market.market_data import (
    MarketDataSnapshot,
    EquitySpot,
    FixedRate,
    StaticVolatility,
    DividendRate,
)
from atlas.domain.enums import Currency
from atlas.compute.pricing.pricing_dispatcher import calculate_price

# 1. Setup Dates
valuation_date = ql.Date(22, 6, 2026)
expiry_date = ql.Date(22, 6, 2027) # 1 year to maturity

# 2. Construct the Option Instrument
option = EuropeanEquityOption(
    strike_price=100.0,
    expiry_date=expiry_date,
    equity_ticker="AAPL",
    option_type=1,  # 1 for Call option, -1 for Put option
    currency=Currency.USD,
)

# 3. Construct the Market Data Snapshot
market_data = MarketDataSnapshot(
    valuation_date=valuation_date,
    equity_spots={
        "AAPL": EquitySpot(equity_ticker="AAPL", currency=Currency.USD, value=100.0)
    },
    fixed_rate={
        Currency.USD: FixedRate(rate=0.05)  # 5% risk-free interest rate
    },
    volatility_rates={
        "AAPL": StaticVolatility(volatility=0.2)  # 20% volatility
    },
    dividend_rates={
        "AAPL": DividendRate(rate=0.0)  # 0% dividend yield
    }
)

# 4. Price the Option using the Dispatcher
option_price_dispatcher = calculate_price(option, market_data)
print(f"European Option Price (via Dispatcher): {option_price_dispatcher:.6f}")

European Option Price (via Dispatcher): 10.450584


## Method 2: Direct Black-Scholes-Merton Pricing

In this method, we directly call the low-level `black_scholes_merton_pricer` with the exact same inputs as the instrument above to verify the pricing result.

In [2]:
from atlas.compute.pricing.equities.equity_option_pricers import black_scholes_merton_pricer

# Price the Option directly
option_price_direct = black_scholes_merton_pricer(
    valuation_date=valuation_date,
    expiry_date=expiry_date,
    spot_price=100.0,
    strike_price=100.0,
    risk_free_rate=0.05,
    dividend_yield=0.0,
    volatility=0.2,
    option_type=1,
)
print(f"European Option Price (Direct BSM):    {option_price_direct:.6f}")
print(f"Verification (Prices Match):           {option_price_dispatcher == option_price_direct}")

European Option Price (Direct BSM):    10.450584
Verification (Prices Match):           True
